# 06b — LIANA: Permutation-Based Significance (audit re-run)

**Stage 5 audit · do the headline LR pairs survive a real significance test?**

This is an **audit re-analysis, not a replacement.** `06_liana_cellcell_communication.ipynb`
stays as the original record with its original bubble plots and `delta_rank` tables; this
notebook reruns the identical pipeline with `n_perms=100` instead of `None`.

**Inputs**
- `data/Cherief_scRNA-seq/GSE244921_processed.h5ad` — from `01`
- `data/Cherief_scRNA-seq/GSE244921_cluster8_sub.h5ad` — from `02`

**Outputs**
- `data/liana/liana_differential_stromal_v2_permtested.csv` — differential table carrying `cellphone_pvals` and `cellchat_pvals`

**Runs after:** `06` · **Feeds:** the Stage 5 caveats carried into `07`

**Runtime:** ~8–10 minutes for both conditions at `n_perms=100`.

**Environment:** analysis env (`environments/analysis.txt`). Run with the working directory set to `scripts/ipynb/` — every path below is relative to it.

---

**Why this notebook exists:** `06_liana_cellcell_communication.ipynb` called `li.mt.rank_aggregate(..., n_perms=None)`, explicitly skipping permutation testing and using only the aggregate `magnitude_rank` (a rank-based consensus score across 6 methods, not a p-value). The headline results table in `docs/progress.md` ("signals STRONGER in innervated", `delta_rank +0.94`, etc.) was built entirely from `delta_rank` — the arithmetic difference of two independently-computed ranks — with no significance test behind it anywhere. Setting `n_perms=100` activates the CellPhoneDB-style and CellChat-style permutation p-values (`cellphone_pvals`, `cellchat_pvals`) that were previously disabled.

**Important scope limitation, stated up front:** permutation testing here shuffles cell-type labels *within one condition's pooled sample* — it tests "is this ligand-receptor pair's signal more specific to this sender→receiver pair than random relabeling," not "does this differ between biological replicates of innervated vs. denervated animals." Cherief 2023 is one pooled sample per condition (no per-animal replicate data recoverable from the GEO deposit). So this fixes the "delta_rank isn't a significance test" problem, but it does **not** and cannot fix the deeper pseudoreplication problem (see Stage 4/5 audit notes) — that remains a caveat to state, not something any local re-analysis can resolve.


## 0. Setup

Imports, paths and output directories.


In [1]:
import time
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import scipy.sparse as sp
import liana.method._liana_pipe as _lp
import liana.method.sc._cellchat as _cc
from pathlib import Path

sc.settings.verbosity = 1

ROOT    = Path('../..')
DATA_SC = ROOT / 'data' / 'Cherief_scRNA-seq'
DATA_LI = ROOT / 'data' / 'liana'
FIG     = ROOT / 'figures' / 'liana'

print('liana', li.__version__)

liana 0.1.9


## 1. Rebuild the identical annotated dataset (same code as §1 of `06_liana_cellcell_communication.ipynb`)

In [2]:
adata_full = sc.read(DATA_SC / 'GSE244921_processed.h5ad')

cluster_map = {
    '0':  'Macrophage', '1':  'Macrophage', '2':  'Macrophage', '3':  'Immune_other',
    '4':  'Tenocyte', '5':  'Tenocyte', '6':  'Tenocyte', '7':  'Stromal_other',
    '8':  'PDGFRa_stromal', '9':  'Smooth_muscle', '10': 'Endothelial', '11': 'Immune_other',
    '12': 'Endothelial', '13': 'Macrophage', '14': 'Immune_other', '15': 'Stromal_other',
    '16': 'Immune_other',
}
adata_full.obs['cell_type'] = adata_full.obs['leiden'].map(cluster_map).astype('category')

sub = sc.read(DATA_SC / 'GSE244921_cluster8_sub.h5ad')
sub_labels = sub.obs['cell_type']
adata_full.obs['cell_type'] = adata_full.obs['cell_type'].astype(str)
common = adata_full.obs.index.intersection(sub_labels.index)
adata_full.obs.loc[common, 'cell_type'] = sub_labels.loc[common].astype(str)
adata_full.obs['cell_type'] = adata_full.obs['cell_type'].astype('category')

keep = {'Macrophage', 'Tenocyte', 'TSPC', 'T-FAP', 'Tenogenic-progenitor',
        'Stromal', 'Endothelial', 'Smooth_muscle', 'Stromal_other'}
mask = adata_full.obs['cell_type'].isin(keep)
adata = adata_full[mask].copy()
print(f'Retained {adata.n_obs} / {adata_full.n_obs} cells')
print(adata.obs['cell_type'].value_counts())

Retained 20047 / 22615 cells
cell_type
Tenocyte                6795
Macrophage              5183
Endothelial             1740
Smooth_muscle           1417
Stromal                 1388
T-FAP                   1245
Stromal_other           1012
Tenogenic-progenitor     816
TSPC                     451
Name: count, dtype: int64


## 2. Same liana 0.1.9 compatibility patches as the original notebook

In [3]:
def _trimean_patched(a, axis=0):
    arr = a.toarray() if sp.issparse(a) else np.asarray(a)
    return np.mean(np.quantile(arr, q=[0.25, 0.5, 0.5, 0.75], axis=axis), axis=axis)
_lp._trimean = _trimean_patched

def _lr_probability_patched(perm_stats, axis=0):
    lr_prob = np.prod(perm_stats, axis=axis)
    return lr_prob / (0.5 + lr_prob)
_cc._lr_probability = _lr_probability_patched
print('patches applied')

patches applied


## 3. Rerun `rank_aggregate` with `n_perms=100` (was `None`)

Identical call to the original notebook except for `n_perms`. A quick timing calibration (n_perms=10 on the denervated subset) took ~24s, so n_perms=100 on both conditions is expected to take roughly 8-10 minutes total.

In [4]:
conditions = {'innervated': 'TrkAWT', 'denervated': 'TrkAF592A'}
liana_results_v2 = {}

for label, cond_val in conditions.items():
    print(f'\n=== Running LIANA (n_perms=100): {label} ({cond_val}) ===')
    t0 = time.time()
    sub_c = adata[adata.obs['condition'] == cond_val].copy()
    print(f'  {sub_c.n_obs} cells')

    li.mt.rank_aggregate(
        sub_c,
        groupby='cell_type',
        resource_name='mouseconsensus',
        expr_prop=0.1,
        min_cells=10,
        use_raw=True,
        n_perms=100,
        verbose=False,
        inplace=True,
    )
    liana_results_v2[label] = sub_c.uns['liana_res'].copy()
    print(f'  -> {len(liana_results_v2[label])} LR pairs scored in {time.time()-t0:.0f}s')


=== Running LIANA (n_perms=100): innervated (TrkAWT) ===
  11478 cells


c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\legacy_api_wrap\__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\indexing.py:1858: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\liana\method\_pipe_utils\_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\liana\method\_liana_pipe.py:312: ImplicitModific

  -> 56701 LR pairs scored in 46s

=== Running LIANA (n_perms=100): denervated (TrkAF592A) ===
  8569 cells


c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\legacy_api_wrap\__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\indexing.py:1858: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\liana\method\_pipe_utils\_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
c:\Users\fangy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\liana\method\_liana_pipe.py:312: ImplicitModific

  -> 57233 LR pairs scored in 39s


## 4. Rebuild the same differential table, now carrying permutation p-values through

In [5]:
key_cols = ['source', 'target', 'ligand_complex', 'receptor_complex']
value_cols = ['magnitude_rank', 'cellphone_pvals', 'cellchat_pvals']

inn = liana_results_v2['innervated'][key_cols + value_cols].rename(
    columns={c: f'{c}_innervated' for c in value_cols})
den = liana_results_v2['denervated'][key_cols + value_cols].rename(
    columns={c: f'{c}_denervated' for c in value_cols})

diff_v2 = inn.merge(den, on=key_cols, how='inner')
diff_v2['delta_rank'] = diff_v2['magnitude_rank_denervated'] - diff_v2['magnitude_rank_innervated']
print(f'LR pairs present in both conditions: {len(diff_v2)}')

diff_v2_stromal = diff_v2[diff_v2['target'].isin(['TSPC', 'T-FAP'])].copy()
print(f'LR pairs with TSPC or T-FAP as receiver: {len(diff_v2_stromal)}')

LR pairs present in both conditions: 52646
LR pairs with TSPC or T-FAP as receiver: 11669


## 5. Do the specific headline pairs from Stage 5 actually hold up under permutation testing?

For each headline pair, check `cellphone_pvals` (permutation p-value; lower = more specific than random cell-label shuffling) **in the condition where the original notebook claimed the signal was strong**. E.g. for "stronger in innervated" pairs, is `cellphone_pvals_innervated < 0.05`?

In [6]:
headline_innervated = [
    ('Stromal_other', 'T-FAP', 'Efna1', 'Epha3'),
    ('Stromal_other', 'T-FAP', 'Sema4a', 'Plxnd1'),
    ('TSPC', 'TSPC', 'Bmp3', 'Bmpr1b'),
    ('Smooth_muscle', 'TSPC', 'Wnt5a', 'Fzd4'),
    ('Stromal_other', 'TSPC', 'Pdgfc', 'Pdgfra'),
]
headline_denervated = [
    ('TSPC', 'TSPC', 'Sema3f', 'Nrp2_Plxna1'),
    ('Macrophage', 'TSPC', 'Tnf', 'Notch1'),
]

def check_pairs(pairs, direction):
    rows = []
    for source, target, ligand, receptor in pairs:
        match = diff_v2[(diff_v2['source'] == source) & (diff_v2['target'] == target) &
                         (diff_v2['ligand_complex'] == ligand) & (diff_v2['receptor_complex'].str.contains(receptor, regex=False))]
        if match.empty:
            rows.append({'signal': f'{ligand}->{receptor}', 'source': source, 'target': target, 'found': False})
            continue
        r = match.iloc[0]
        col = 'cellphone_pvals_innervated' if direction == 'innervated' else 'cellphone_pvals_denervated'
        rows.append({
            'signal': f'{ligand}->{receptor}', 'source': source, 'target': target, 'found': True,
            'claimed_stronger_in': direction,
            'cellphone_pval_in_claimed_condition': r[col],
            'delta_rank': r['delta_rank'],
        })
    return pd.DataFrame(rows)

res_inn = check_pairs(headline_innervated, 'innervated')
res_den = check_pairs(headline_denervated, 'denervated')
headline_check = pd.concat([res_inn, res_den], ignore_index=True)
headline_check

,signal,source,target,found,claimed_stronger_in,cellphone_pval_in_claimed_condition,delta_rank
0,Efna1->Epha3,Stromal_other,T-FAP,True,innervated,1.0,0.936328
1,Sema4a->Plxnd1,Stromal_other,T-FAP,True,innervated,1.0,0.855546
2,Bmp3->Bmpr1b,TSPC,TSPC,True,innervated,0.0,0.789815
3,Wnt5a->Fzd4,Smooth_muscle,TSPC,True,innervated,1.0,0.752346
4,Pdgfc->Pdgfra,Stromal_other,TSPC,True,innervated,0.0,0.044689
5,Sema3f->Nrp2_Plxna1,TSPC,TSPC,True,denervated,1.0,-0.819437
6,Tnf->Notch1,Macrophage,TSPC,True,denervated,0.0,-0.738222


## 6. Broader calibration: of the original notebook's top-20 "gained"/"lost" tables, how many are permutation-significant?

In [7]:
top20_gained_innervated = diff_v2_stromal.sort_values('delta_rank', ascending=False).head(20)
top20_gained_denervated = diff_v2_stromal.sort_values('delta_rank', ascending=True).head(20)

n_sig_inn = (top20_gained_innervated['cellphone_pvals_innervated'] < 0.05).sum()
n_sig_den = (top20_gained_denervated['cellphone_pvals_denervated'] < 0.05).sum()
print(f'Top-20 "stronger in innervated" pairs: {n_sig_inn}/20 have cellphone_pvals_innervated < 0.05')
print(f'Top-20 "stronger in denervated" pairs: {n_sig_den}/20 have cellphone_pvals_denervated < 0.05')

diff_v2_stromal.to_csv(DATA_LI / 'liana_differential_stromal_v2_permtested.csv', index=False)
print('Saved: data/liana/liana_differential_stromal_v2_permtested.csv')

Top-20 "stronger in innervated" pairs: 8/20 have cellphone_pvals_innervated < 0.05
Top-20 "stronger in denervated" pairs: 12/20 have cellphone_pvals_denervated < 0.05
Saved: data/liana/liana_differential_stromal_v2_permtested.csv


## 7. Conclusion for Stage 5/6

Report the actual numbers from the cells above rather than assuming — but the key methodological fix stands regardless of the exact counts: **the original "signals STRONGER in innervated/denervated" tables were built on `delta_rank` (a rank-difference heuristic) with permutation testing explicitly disabled.** With `n_perms=100`, real permutation p-values (`cellphone_pvals`) are now available and should be used as the primary evidence for which specific LR pairs are trustworthy, with `delta_rank` treated as a secondary/descriptive ranking rather than the headline statistic.

**This does not resolve the deeper limitation:** both conditions are still single pooled samples (no biological replicates), so even a low `cellphone_pvals` only says "this pattern is unlikely under random cell relabeling within this one sample" — not "this would replicate in a second cohort of mice." That caveat belongs in the report regardless of how many pairs pass permutation testing here.